# Experiments without downloading data

This notebook runs the independent re-execution and the revision experiments with every
input streamed from its public URL into memory. Nothing is written to `data/`; the only
files written are the result CSVs under `experiments/results/`.

Set `MODELS` below to `"five"` (the five earlier graph networks, files hosted on GitHub) or
`"all"` (plus the ten current-generation models, files hosted on figshare). Start the
notebook from the repository root or from `notebooks/`.

In [ ]:
import os, pathlib, subprocess, sys
import pandas as pd

MODELS = "all"          # "five" or "all"

os.environ["WBM_DATA_SOURCE"] = "url"   # stream every input; no files on disk
os.environ["MODELS"] = MODELS
here = pathlib.Path.cwd()
root = here.parent if here.name == "notebooks" else here
EXP = root / "experiments"
assert (EXP / "wbm_data.py").is_file(), "start this notebook from the repository root or notebooks/"
print("repository:", root, "| data source: streamed | models:", MODELS)

def run(script):
    """Run one experiment script in experiments/ and echo its output."""
    print(f"$ MODELS={MODELS} WBM_DATA_SOURCE=url python {script}")
    r = subprocess.run([sys.executable, script], cwd=EXP, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-3000:]); raise RuntimeError(f"{script} failed")

## 1. Census reproduction (Table 1 columns from the public prediction files)

In [ ]:
run("exp0_census_reproduction.py")
pd.read_csv(EXP / "results/exp0_census_reproduction.csv").head()

## 2. Re-execution of the run of record (coverage campaigns and paired comparisons)

Expected with `MODELS="all"`: 90 of 90 configurations and 28 of 28 pair rows identical.

In [ ]:
run("exp5_reproduce_e1_e3.py")
suf = "_15models" if MODELS == "all" else ""
e1 = pd.read_csv(EXP / f"results/exp5_e1_reproduction{suf}.csv")
print("E1 identical rows:", int(((e1.coverage == e1.paper_coverage) & (e1.ci_lo == e1.paper_lo) & (e1.ci_hi == e1.paper_hi)).sum()), "of", len(e1))
pd.read_csv(EXP / f"results/exp5_e3_reproduction{suf}.csv")

## 3. Minimal verification budgets (Supplementary Table 5)

In [ ]:
run("exp1_oracle_budgets.py")
pd.read_csv(EXP / "results/exp1_oracle_budgets.csv")

## 4. Transfer of benchmark precision to restricted pools (Supplementary Table 6)

In [ ]:
run("exp2_benchmark_transfer.py")
pd.read_csv(EXP / "results/exp2_benchmark_transfer_arity.csv")

## 5. Cost of a verification sample under size proxies (Supplementary Table 8)

In [ ]:
run("exp3_cost_proxy.py")
pd.read_csv(EXP / "results/exp3_cost_proxy.csv")

## 6. One shared verification sample for several model comparisons (Supplementary Table 7)

About one minute for five models; several minutes for fifteen.

In [ ]:
run("exp4_shared_budget.py")
pd.read_csv(EXP / f"results/exp4_shared_budget{suf}.csv").pivot_table(index="pair", columns=["design", "total_labels"], values="holm_power").round(3)

## 7. Reconcile the paper with the result files

In [ ]:
r = subprocess.run([sys.executable, "scripts/verify_results.py", "results", "experiments/results"], cwd=root, capture_output=True, text=True)
print(r.stdout[-1500:]); assert r.returncode == 0, "verification failed"